# 01. Data Quality

This notebook runs the reusable GiftBridge CSV validator against the generated development fixture. The input is synthetic by design and is not evidence about the college, donors, or villages. When your own pilot data is collected, validate its protected export separately with `expected_origin="real"`.

In [ ]:
from pathlib import Path
from ml.data.quality import validate_dataset

cwd = Path.cwd()
WORKSPACE = next((path for path in (cwd, *cwd.parents) if (path / "ml" / "data" / "quality.py").exists()), cwd)
SYNTHETIC_DIR = WORKSPACE / "data" / "synthetic"
report = validate_dataset(SYNTHETIC_DIR, expected_origin="synthetic")
print(f"Checked dataset: {SYNTHETIC_DIR}")
print(f"Passed: {report['passed']}")
print("Rows by table:")
for table, count in report["row_counts"].items():
    print(f"  {table}: {count}")
print("Issues:")
if report["issues"]:
    for issue in report["issues"]:
        print(f"  {issue['severity'].upper()} [{issue['table']}]: {issue['message']}")
else:
    print("  None")

In [ ]:
# Plain-language conclusion
if report["passed"]:
    print(f"Conclusion: all {len(report['row_counts'])} required synthetic tables passed the current structural, range, relationship, and provenance checks.")
else:
    error_count = sum(issue["severity"] == "error" for issue in report["issues"])
    print(f"Conclusion: validation found {error_count} error(s). Resolve them before analysis or import.")
print("Limitations: these checks cannot establish consent, truth, representativeness, or image/text safety. Synthetic rows must not be included in impact claims.")